# 02 — H1a formal test (ADR 0006)

**Protocol:** `docs/decisions/0006-h1-formal-testing-protocol.md`, frozen at commit `ede98e0`
before any model was estimated on real data. Nothing here may change a specification; any
analysis not listed in the ADR must be labelled *post-hoc* and recorded in a new ADR.

Every estimate comes from the validated, tested functions in `src/kagura/models.py`,
assembled by `src/kagura/h1a.py`. The notebook only calls them and displays results.

| Label | Meaning (ADR 0006) | Sections |
|---|---|---|
| **CONFIRMATORY** | Decides H1a: primary specification, gate, S and R, α = 0.025 one-sided | 4–6 |
| **SECONDARY** | Formal test, unadjusted α = 0.05, never decides H1a | 7 |
| **DESCRIPTIVE** | No inference beyond what is printed; never decides H1a | 8–9 |
| **ROBUSTNESS** | R1–R3 re-run the §5.1 rule; they label the primary result, never change it | 10 |

Re-run from the repo root (only after the data integrity check passes):

```bash
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/02_h1a_formal.ipynb
```

## 1. Research question and frozen methodology

**H1a (short-run comovement).** Has the contemporaneous association between daily USD/JPY
changes and daily changes in the US–Japan 2Y yield differential declined *materially*
between the reference era 2013–2021 and the candidate new era from 2022-01-01?

The 2022-01-01 boundary is a pre-specified **candidate** regime boundary, not an assumed
break. β is an association: it is not a causal effect and not a forecast (ADR 0006 §3).
The write-up uses association language only ("is associated with", "comoves with").

**Regression (§4.3).** Fully era-interacted OLS over the five ADR 0004 eras, Newey–West HAC
covariance, lag `floor(4·(T/100)^(2/9))`, no small-sample correction:

    Δs_t = Σ_e 1[t ∈ e] · (α_e + β_e · Δd_t) + ε_t

**Test quantity (§5).** δ = β_new − (2/3)·β_ref. A decline of at least ⅓ (the preselected
operational definition of materiality) means δ < 0.

**Decision rules (§5.1),** α = 0.025 one-sided, c = Φ⁻¹(0.975):

| Check | Rule | Outcome |
|---|---|---|
| G (gate) | L_ref = β̂_ref − c·se > 0 | fails → **not assessable** |
| S | U = δ̂ + c·se < 0 | **supported** (material decline) |
| R | L = δ̂ − c·se > 0 | **ruled out** (decline smaller than ⅓, or an increase; *not* "no change") |
| neither | — | **inconclusive** (not evidence of stability, not evidence of decline) |

At the boundary δ = 0 the combined probability of either directional conclusion can reach
0.05. Disclosure (§1.1): exploratory statistics already implied β_ref ≈ 6.3 and
β_new ≈ 4.0 (ratio ≈ 0.63); this test is pre-specified, not blind.

In [ ]:
import pandas as pd
from IPython.display import display

from kagura import explore as ex
from kagura import h1a
from kagura import models as m

frozen = {
    "alpha (one-sided, gate/S/R)": m.ALPHA,
    "c = Phi^-1(1 - alpha)": m.C,
    "material decline": m.MATERIAL,
    "sensitivity declines (reported only)": m.SENSITIVITY,
    "reference era": m.REF,
    "new era (candidate boundary 2022-01-01)": m.NEW,
    "daily sample end": m.DAILY_END,
    "monthly sample end": m.MONTHLY_END,
    "secondary alpha (unadjusted)": m.WALD_ALPHA,
}
for k, v in frozen.items():
    print(f"{k:42s} {v}")

## 2. Data sources, versions and integrity verification

Sources are recorded in ADR 0002 and the alignment rules in ADR 0003. The analysis runs only
on the frozen vintage of ADR 0006 §9.2: `load_verified` checks the SHA-256 of every listed
file, then the schemas, dates and sample boundaries, and raises `IntegrityError` before
returning any data if anything differs. Estimation (`h1a.analyze`) accepts only the
`VerifiedData` object this cell returns.

In [ ]:
# Stops here (IntegrityError) on any checksum, schema or boundary problem. nbconvert halts
# on the first error, so no later cell, and no regression, can run on unverified data.
data = h1a.load_verified()
daily, monthly = data.daily, data.monthly
for name, df, fmt in (("daily", daily, "%Y-%m-%d"), ("monthly", monthly, "%Y-%m")):
    first, last = df["date"].min(), df["date"].max()
    print(f"{name:8s} {first:{fmt}} .. {last:{fmt}}  rows={len(df)}")
display(h1a.vintage_table())

## 3. Sample construction and exclusions

Daily sample 1981-01-01 to 2026-09-11; monthly 1981-01 to 2026-08 (§9.2). A daily row is used
only if both changes are between consecutive live observations (ADR 0004 exclusion rule);
the remaining rows are treated as consecutive for HAC (assumption A9). No trimming or
winsorising. Eras are assigned by each change's own date (its end date).

`h1a.analyze` runs every ADR 0006 analysis once, in protocol order; the sections below only
display parts of it. Row counts per era are reported for every specification; the expected
counts in the ADR were provisional.

In [ ]:
analysis = h1a.analyze(data)  # every regression in this notebook runs here, once
specs = h1a.specifications(data.daily, data.monthly)  # change tables, for the rolling plot

print("Exclusion rule, rows with a differential from 1981 (ADR 0004):")
display(analysis.exclusions)
print("Rows used per era, by specification:")
display(pd.DataFrame({spec: fit.n for spec, fit in analysis.fits.items()}))

## 4. Primary H1a regression — CONFIRMATORY

Specification: daily, 2Y differential, same-day timing (§4.1). Coefficients in % change of
USD/JPY per 1 pp change in the differential; HAC standard errors.

In [ ]:
fit = analysis.fits["primary"]
coef = pd.DataFrame({"estimate": fit.params, "hac_se": fit.cov.to_numpy().diagonal() ** 0.5})
display(coef)
print(f"rows used: {fit.nobs}   HAC lag: {fit.lag}")
print("R² by era (descriptive):")
display(fit.r2)

## 5. Positivity gate and directional tests S and R — CONFIRMATORY

Rules exactly as in §5.1 (see section 1). The outcome below is the H1a result; nothing in
sections 7–10 can change it.

In [ ]:
r = analysis.primary
assert r.spec == "primary"
b, d = r.beta_ref, r.delta
print(f"G  H0 beta_ref <= 0: L_ref = {b.lower: .4f}  p = {b.p_above:.4g}  passes: {r.gate}")
print(f"S  H0 delta >= 0:    U = {d.upper: .4f}  p_S = {d.p_below:.4g}  rejects: {r.s_rejects}")
print(f"R  H0 delta <= 0:    L = {d.lower: .4f}  p_R = {d.p_above:.4g}  rejects: {r.r_rejects}")
print(f"\nH1a OUTCOME (confirmatory): {r.outcome}")

## 6. Coefficient estimates, confidence intervals and effect sizes — CONFIRMATORY (reporting)

Intervals are two-sided 95%; each endpoint is the one-sided 97.5% bound used by the rules.
The relative change uses a Fieller confidence set, which may be unbounded and is never
truncated (§5.2). The ¼ and ½ thresholds are sensitivity only.

In [ ]:
display(h1a.effects_table(r))
print(f"relative change beta_new/beta_ref - 1: {r.relative_change: .4f}")
print(f"Fieller 95% set for the relative change: {h1a.format_set(r.relative_change_set)}")
print(f"implied % USD/JPY association with a 100 bp narrowing: {r.implied_100bp_narrowing}")
print(f"R²: {r.r2}")

## 7. Era-equality Wald test — SECONDARY

§6.1: HAC Wald test of H0: β equal in all five eras, χ²(4), unadjusted α = 0.05. It answers
whether one β describes the whole sample. It does not decide H1a.

In [ ]:
w = analysis.era_equality
print(f"Wald = {w.stat:.4f}  df = {w.df}  p = {w.p:.4g}  rejects at 0.05 (unadjusted): {w.rejects}")
display(coef.loc[[f"beta:{e}" for e in fit.n.index]])

## 8. Rolling beta — DESCRIPTIVE

§6.3: trailing 250- and 750-row windows of the USD/JPY calendar, reported only when at least
80% of the window's rows survive the exclusion rule; lines break where a window is not
reported. No inference: the plot shows drift, not tests.

In [ ]:
fig = m.plot_rolling_beta(specs["primary"])
ex.save(fig, "h1a_rolling_beta")

## 9. Two-leg US/Japan decomposition — DESCRIPTIVE

§6.3: Δs = α + β_US·Δus + β_JP·Δjgb + ε by era, with HAC Wald tests of β_US = −β_JP
(equivalently β_US + β_JP = 0). Where a leg is constant in an era its slope is not
identified and no test is reported. A non-rejection with a wide interval reflects
imprecision (the JGB leg was nearly pinned in 1999–2022), not evidence that the legs offset.

In [ ]:
rows = {}
for era, t in analysis.legs.items():
    rows[era] = {"identified": t.identified}
    if t.identified and t.total is not None and t.wald is not None:
        rows[era] |= {
            "beta_us": analysis.two_leg.params[f"beta_us:{era}"],
            "beta_jp": analysis.two_leg.params[f"beta_jp:{era}"],
            "beta_us + beta_jp": t.total.estimate,
            "lower": t.total.lower,
            "upper": t.total.upper,
            "wald": t.wald.stat,
            "p": t.wald.p,
        }
display(pd.DataFrame(rows).T)
print("Leg spread (diagnostic):")
display(analysis.leg_spread)

## 10. Robustness R1–R3 — ROBUSTNESS

§9.1: each variant re-runs the §5.1 rule with one change. R1: lagged US leg. R2: monthly
month-end changes (low power). R3: 10Y differential. The label is "robust" only if all
three share the primary category; robustness never changes the primary outcome.

In [ ]:
display(h1a.results_table(analysis.results))
print(f"Primary outcome: {analysis.primary.outcome}   Robustness label: {analysis.robustness}")

## 11. Statistical interpretation and limitations

**Interpretation: to be written in Stage 3B**, after this notebook has been executed and its
output reviewed. It must use association language (§3), report the outcome category
exactly as computed in section 5, and give absolute and relative changes with intervals.

**Limitations fixed in advance by ADR 0006:**

- Pre-specified, not blind (§1.1): exploratory statistics implied the coefficients.
- Timing: the same-day β is a projection coefficient whose distortion may differ between
  eras (A5, §4.5); R2 is the guard, intraday data could settle it.
- β_ref comes from the quietest-differential era, so its standard error is relatively large.
- Normal approximation (A7); the slight liberal bias of Newey–West without correction was
  measured on synthetic data in Stage 1 review.
- The 2022-01 boundary is a candidate: if mis-dated, the contrast is biased toward no change.
- Unknown-break-date inference and H1b are deferred to separate ADRs.

## 12. Reproducibility information

In [ ]:
for k, v in h1a.provenance().items():
    print(f"{k:14s} {v}")
print("data checksums: verified against ADR 0006 §9.2 in section 2")
print("randomness: none (no bootstrap in H1a)")